# FCC metal surfaces with ASE and TBLite

Generate flat or stepped FCC metal surfaces directly from a metal and macroscopic Miller indices, for example Pt(211). The generator automatically identifies the supported Lang-Joyner-Somorjai (LJS) terrace and step geometry. ASE builds a minimal FCC surface cell using its reference lattice parameter by default and handles constraints, visualization, optimization, and output with TBLite energies and forces. No external structure or calculator input files are needed.

Install the Python dependencies in the notebook environment if needed: `ase`, `tblite`, and `numpy`. Isometric previews additionally require the external `povray` executable on `PATH`.

In [ ]:
### Uncomment if using Colab
!apt install povray povray-includes
!pip install ase tblite

In [ ]:
from fractions import Fraction
from functools import reduce
from itertools import permutations
from math import ceil, gcd
from numbers import Integral
from pathlib import Path
from shutil import which
from subprocess import run

import numpy as np

from ase.build import bulk, make_supercell, surface
from ase.constraints import FixAtoms
from ase.data import atomic_numbers
from ase.geometry import minkowski_reduce
from ase.io import read, write
from ase.optimize import BFGS
from ase.visualize import view
from IPython.display import Image as NotebookImage, display
from tblite.ase import TBLite

---
## LJS notation, zone axes, and Miller indices

For a metal <i>M</i>, the LJS notation below describes the local geometry of a stepped single-crystal surface:

M(S)-[<i>n</i>(<i>hkl</i>) &times; (<i>h&prime;k&prime;l&prime;</i>)]

- M(S) denotes the macroscopic stepped surface.
- (<i>hkl</i>) before the multiplication sign is the terrace microfacet.
- <i>n</i> is the number of atomic rows across that terrace.
- (<i>h&prime;k&prime;l&prime;</i>) after the multiplication sign is the monatomic step microfacet.
- The zone axis [<i>uvw</i>] is the common direction of the terrace and step planes and is therefore parallel to the step edge.

The facet indices inside the LJS symbol describe the **local** terrace and step; they are not the Miller indices of the complete stepped surface. The macroscopic Miller indices are obtained from the corresponding formula in the table. A barred index is negative: for example, <span style="text-decoration: overline;">1</span> = &minus;1. The forms containing (11<span style="text-decoration: overline;">1</span>) select planes whose intersection is the listed [1<span style="text-decoration: overline;">1</span>0] step direction; the function uses the unsigned cubic facet-family labels `"111"`, `"100"`, and `"110"`.

| Zone axis | LJS notation | Miller formula | Example after reduction |
| :--- | :--- | :--- | :--- |
| [1<span style="text-decoration: overline;">1</span>0] | M(S)-[(<i>n</i>&minus;1)(111) &times; (110)] | (<i>n</i>, <i>n</i>, <i>n</i>&minus;2) | <i>n</i> = 4: 3(111) &times; (110) = M(221) |
| [1<span style="text-decoration: overline;">1</span>0] | M(S)-[<i>n</i>(111) &times; (11<span style="text-decoration: overline;">1</span>)] | (<i>n</i>, <i>n</i>, <i>n</i>&minus;2) | <i>n</i> = 4: (4, 4, 2) &rarr; (2, 2, 1) = M(221) |
| [01<span style="text-decoration: overline;">1</span>] | M(S)-[<i>n</i>(111) &times; (100)] | (<i>n</i>+1, <i>n</i>&minus;1, <i>n</i>&minus;1) | <i>n</i> = 3: (4, 2, 2) &rarr; (2, 1, 1) = M(211) |
| [001] | M(S)-[<i>n</i>(100) &times; (110)] | (<i>n</i>, 1, 0) | <i>n</i> = 3: (3, 1, 0) = M(310) |
| [1<span style="text-decoration: overline;">1</span>0] | M(S)-[<i>n</i>(110) &times; (11<span style="text-decoration: overline;">1</span>)] | (2<i>n</i>&minus;1, 2<i>n</i>&minus;1, 1) | <i>n</i> = 3: (5, 5, 1) = M(551) |
| [01<span style="text-decoration: overline;">1</span>] | M(S)-[<i>n</i>(100) &times; (111)] | (2<i>n</i>&minus;1, 1, 1) | <i>n</i> = 3: (5, 1, 1) = M(511) |
| [001] | M(S)-[<i>n</i>(110) &times; (100)] | (<i>n</i>, <i>n</i>&minus;1, 0) | <i>n</i> = 3: (3, 2, 0) = M(320) |

### How the Miller tuple is normalized

For a cubic crystal, a plane (<i>hkl</i>) is normal to the reciprocal-lattice vector

<b>G</b><sub><i>hkl</i></sub> = <i>h</i><b>a</b><sup>*</sup> + <i>k</i><b>b</b><sup>*</sup> + <i>l</i><b>c</b><sup>*</sup>.

A zero index means that the plane is parallel to that crystallographic axis, while an overbar denotes a negative intercept. Integer triples that differ only by a common factor describe the same orientation, so input Miller indices are reduced to the smallest integer triple before ASE constructs the surface and the LJS formulas are matched. The reduced tuple is stored as `miller_index`.

For the worked Pt example, reduction gives

Pt(S)[3(111) &times; (100)] &rarr; (4, 2, 2) &rarr; (2, 1, 1) = Pt(211).

The same mapping applies to every FCC metal.

### Inferring LJS notation from Miller indices

`generate_slab` works in the reverse direction: it reduces and canonicalizes the requested cubic Miller indices, tests the formulas in the table, and records every matching terrace/step description. Some macroscopic orientations have more than one valid local description. The function uses the table order as its naming convention and preserves all matches in `atoms.info["surface"]["ljs_candidates"]`. Thus a surface in the (<i>n</i>, <i>n</i>, <i>n</i>&minus;2) series is reported first as M(S)[(<i>n</i>&minus;1)(111) &times; (110)], matching the convention used for the render series below. For example:

```python
pt221 = generate_slab("Pt", 2, 2, 1)
print(pt221.info["surface"]["ljs_candidates"])
```

For Pt(211), the selected result is three-row (111) terraces with (100) steps. Flat (111), (100), and (110) inputs have no step facet. Indices outside the supported LJS families raise a clear `ValueError` instead of assigning a misleading terrace/step geometry.

In [ ]:
# Ordered by the preferred description when one orientation has several LJS names.
LJS_FAMILIES = {
    ("111", "110"): {"zone_axis": (1, -1, 0), "formula": "(n+1, n+1, n-1)"},
    ("111", "100"): {"zone_axis": (0, 1, -1), "formula": "(n+1, n-1, n-1)"},
    ("111", "111"): {"zone_axis": (1, -1, 0), "formula": "(n, n, n-2)"},
    ("100", "110"): {"zone_axis": (0, 0, 1), "formula": "(n, 1, 0)"},
    ("110", "111"): {"zone_axis": (1, -1, 0), "formula": "(2n-1, 2n-1, 1)"},
    ("100", "111"): {"zone_axis": (0, 1, -1), "formula": "(2n-1, 1, 1)"},
    ("110", "100"): {"zone_axis": (0, 0, 1), "formula": "(n, n-1, 0)"},
}


def _reduce_miller(indices):
    divisor = reduce(gcd, (abs(value) for value in indices))
    return tuple(value // divisor for value in indices)


def _raw_ljs_miller(n, family):
    """Evaluate one of the Miller-index formulas in LJS_FAMILIES."""
    if family == ("111", "111"):
        return n, n, n - 2
    if family == ("111", "110"):
        return n + 1, n + 1, n - 1
    if family == ("111", "100"):
        return n + 1, n - 1, n - 1
    if family == ("100", "110"):
        return n, 1, 0
    if family == ("110", "111"):
        return 2 * n - 1, 2 * n - 1, 1
    if family == ("100", "111"):
        return 2 * n - 1, 1, 1
    if family == ("110", "100"):
        return n, n - 1, 0
    raise ValueError(f"Unknown LJS family: {family}.")

---
## Reusable FCC surface generator

Use one function for both flat and stepped surfaces: `generate_slab("Pt", 2, 1, 1, vacuum=10, width=12, fix=4)`. The first four arguments specify Pt(211). The function recognizes the corresponding LJS family and records its terrace, step, terrace-row count, zone axis, and any equivalent descriptions in `atoms.info["surface"]`. For cubic symmetry-equivalent index permutations it uses the same facet family. If more than one LJS description is possible, it chooses the one with the widest terrace and retains every match in `ljs_candidates`.

`width` is the requested slab thickness in angstrom. `vacuum` is the empty space added independently below and above the outermost atoms. `fix` is a physical depth in angstrom measured upward from the lowest atom; all atoms in that bottom region are constrained, and `fix=0` leaves every atom free. Use `a` only to override ASE's FCC lattice constant. An elemental FCC surface has one translationally unique cut, so `index` is retained as an explicit guard and must be `0`. In-plane repetition is intentionally not part of `generate_slab`; apply it afterward with `larger_slab = atoms.repeat((nx, ny, 1))`. The returned slab has a ∥ +x, b ∥ +y, c ∥ +z and 90° cell angles.

In [ ]:
def _normalize_miller(h, k, l):
    """Validate and reduce three integer Miller indices."""
    values = (h, k, l)
    if any(isinstance(value, bool) or not isinstance(value, Integral) for value in values):
        raise ValueError("h, k, and l must be integers.")
    values = tuple(int(value) for value in values)
    if values == (0, 0, 0):
        raise ValueError("Miller indices cannot all be zero.")
    return _reduce_miller(values)


def _distance(name, value, *, positive=False):
    """Return a validated finite distance in angstrom."""
    if isinstance(value, bool):
        raise ValueError(f"{name} must be a finite number in angstrom.")
    try:
        value = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} must be a finite number in angstrom.") from exc
    if not np.isfinite(value) or value < 0 or (positive and value == 0):
        qualifier = "positive" if positive else "non-negative"
        raise ValueError(f"{name} must be a {qualifier} finite number in angstrom.")
    return value


def _count_atomic_layers(atoms, tolerance):
    """Count distinct z layers without constructing index groups."""
    sorted_z = np.sort(atoms.positions[:, 2])
    return 1 + int(np.count_nonzero(np.diff(sorted_z) > tolerance))


def _normalize_metal(metal):
    if not isinstance(metal, str):
        raise ValueError("metal must be an element symbol such as 'Pt', 'Cu', or 'Al'.")
    symbol = metal.strip().capitalize()
    if atomic_numbers.get(symbol, 0) == 0:
        raise ValueError(f"Unknown element symbol: {metal!r}.")
    return symbol


def _build_fcc_primitive(metal, a):
    """Build ASE's one-atom FCC cell and return the conventional parameter a."""
    if a is not None:
        a = _distance("a", a, positive=True)
    try:
        if a is None:
            conventional = bulk(metal, "fcc", cubic=True)
            a = float(conventional.cell.lengths()[0])
        return bulk(metal, "fcc", a=a), a
    except ValueError as exc:
        if a is None:
            raise ValueError(
                f"ASE has no default FCC lattice parameter for {metal}; "
                "pass a=<angstrom>."
            ) from exc
        raise


def _fcc_primitive_miller(miller):
    """Convert conventional cubic FCC indices to ASE's primitive basis."""
    h, k, l = miller
    return _reduce_miller((k + l, h + l, h + k))


def _orthogonal_inplane_transform(
    cell, search_limit=12, denominator_limit=1000, angle_tolerance=1e-7
):
    """Return the smallest integer transform that makes a and b perpendicular."""
    a, b, _ = np.asarray(cell)
    if abs(np.dot(a, b)) <= angle_tolerance * np.linalg.norm(a) * np.linalg.norm(b):
        return np.eye(3, dtype=int)

    candidates = []
    for p in range(-search_limit, search_limit + 1):
        for q in range(-search_limit, search_limit + 1):
            if (p == 0 and q == 0) or gcd(abs(p), abs(q)) != 1:
                continue

            u = p * a + q * b
            u_dot_a = float(np.dot(u, a))
            u_dot_b = float(np.dot(u, b))
            scale = max(
                np.linalg.norm(u) * np.linalg.norm(a),
                np.linalg.norm(u) * np.linalg.norm(b),
                1.0,
            )
            if abs(u_dot_b) <= angle_tolerance * scale:
                r, s = 0, 1
            else:
                ratio = Fraction(u_dot_a / u_dot_b).limit_denominator(denominator_limit)
                r, s = ratio.denominator, -ratio.numerator

            determinant = p * s - q * r
            if determinant == 0:
                continue
            if determinant < 0:
                r, s = -r, -s
                determinant = -determinant

            v = r * a + s * b
            cosine = abs(np.dot(u, v)) / (np.linalg.norm(u) * np.linalg.norm(v))
            if cosine > angle_tolerance:
                continue

            length_ratio = max(np.dot(u, u), np.dot(v, v)) / min(np.dot(u, u), np.dot(v, v))
            score = (
                determinant,
                length_ratio,
                np.dot(u, u) + np.dot(v, v),
                abs(p) + abs(q) + abs(r) + abs(s),
            )
            transform = np.array([[p, q, 0], [r, s, 0], [0, 0, 1]], dtype=int)
            candidates.append((score, transform))

    if not candidates:
        raise RuntimeError("Could not find a reasonably sized orthogonal in-plane supercell.")

    return min(candidates, key=lambda item: item[0])[1]


def _make_orthogonal_surface(atoms):
    """Reduce the 2D lattice, then construct its smallest rectangular cell."""
    _, reduction = minkowski_reduce(atoms.cell, pbc=(True, True, False))
    if round(np.linalg.det(reduction)) < 0:
        reduction[0] *= -1
    atoms = make_supercell(atoms, reduction, wrap=True)
    orthogonal = _orthogonal_inplane_transform(atoms.cell.array)
    atoms = make_supercell(atoms, orthogonal, wrap=True)
    return atoms, (orthogonal @ reduction)


def _rotate_ab_to_xy(atoms, cleanup_tolerance=1e-12):
    """Rotate atoms and cell together around z so a || +x and b || +y."""
    a = atoms.cell.array[0]
    rotation_degrees = -np.degrees(np.arctan2(a[1], a[0]))
    atoms.rotate(rotation_degrees, "z", center=(0.0, 0.0, 0.0), rotate_cell=True)

    # Remove harmless floating-point remnants such as 1e-15 off-axis components.
    cell = atoms.cell.array.copy()
    cell[np.abs(cell) < cleanup_tolerance] = 0.0
    atoms.set_cell(cell, scale_atoms=False)
    return float(rotation_degrees)


def _check_orthogonal_axis_aligned_cell(atoms, tolerance=1e-7):
    """Verify a || +x, b || +y, c || +z and 90/90/90 cell angles."""
    a, b, c = atoms.cell.array
    scale = max(np.linalg.norm(a), np.linalg.norm(b), np.linalg.norm(c), 1.0)
    aligned = (
        np.allclose(a[1:], 0.0, atol=tolerance * scale)
        and np.allclose(b[[0, 2]], 0.0, atol=tolerance * scale)
        and np.allclose(c[:2], 0.0, atol=tolerance * scale)
        and a[0] > 0
        and b[1] > 0
        and c[2] > 0
    )
    orthogonal = all(
        abs(dot_product) <= tolerance * scale**2
        for dot_product in (np.dot(a, b), np.dot(a, c), np.dot(b, c))
    )
    if not (aligned and orthogonal):
        raise RuntimeError("Expected a || +x, b || +y, c || +z with 90/90/90 cell angles.")


def _map_cubic_direction(reference_miller, requested_miller, direction):
    """Apply the signed axis permutation relating a canonical cubic plane to the input."""
    requested_absolute = tuple(abs(value) for value in requested_miller)
    for order in permutations(range(3)):
        if tuple(abs(reference_miller[index]) for index in order) != requested_absolute:
            continue
        signs = tuple(
            -1 if requested_miller[axis] < 0 else 1
            for axis in range(3)
        )
        return tuple(signs[axis] * direction[order[axis]] for axis in range(3))
    raise RuntimeError("Could not map the canonical cubic direction to the requested Miller axes.")


def _integer_quotient(numerator, denominator):
    """Return an integer quotient >= 2, or None when the ratio is not valid."""
    if denominator <= 0 or numerator % denominator:
        return None
    quotient = numerator // denominator
    return quotient if quotient >= 2 else None


def _surface_geometry_from_miller(miller):
    """Infer a canonical low-index terrace/step description for an FCC surface."""
    canonical = tuple(sorted((abs(value) for value in miller), reverse=True))
    flat_facets = {(1, 1, 1): "111", (1, 0, 0): "100", (1, 1, 0): "110"}
    if canonical in flat_facets:
        terrace = flat_facets[canonical]
        return {
            "kind": "flat",
            "terrace_facet": terrace,
            "step_facet": None,
            "terrace_rows": None,
            "zone_axis": None,
            "miller_formula": None,
            "ljs_notation": None,
            "ljs_candidates": [],
        }

    a, b, c = canonical
    same_high = a == b and a > c
    same_low = a > b and b == c and c > 0
    zero_low = a > b > 0 and c == 0

    # These are the table formulas solved directly for n. No search over n is needed.
    row_counts = {
        ("111", "111"): _integer_quotient(2 * a, a - c) if same_high else None,
        ("111", "110"): _integer_quotient(a + c, a - c) if same_high else None,
        ("111", "100"): _integer_quotient(a + b, a - b) if same_low else None,
        ("100", "110"): _integer_quotient(a, b) if zero_low else None,
        ("110", "111"): _integer_quotient(a + c, 2 * c) if same_high and c else None,
        ("100", "111"): _integer_quotient(a + b, 2 * b) if same_low else None,
        ("110", "100"): a if zero_low and a - b == 1 else None,
    }

    candidates = []
    for family, data in LJS_FAMILIES.items():
        n = row_counts[family]
        if n is None:
            continue
        reference_miller = _reduce_miller(_raw_ljs_miller(n, family))
        terrace, step = family
        candidates.append({
            "terrace_rows": n,
            "terrace_facet": terrace,
            "step_facet": step,
            "zone_axis": _map_cubic_direction(
                reference_miller, miller, data["zone_axis"]
            ),
            "miller_formula": data["formula"],
        })

    if not candidates:
        raise ValueError(
            f"FCC Miller index {miller} is not a flat (111), (100), or (110) surface and "
            "does not match one of the supported LJS terrace/step families."
        )

    # LJS_FAMILIES order resolves alternative descriptions of one orientation.
    selected = candidates[0].copy()
    n = selected["terrace_rows"]
    terrace = selected["terrace_facet"]
    step = selected["step_facet"]
    selected.update({
        "kind": "LJS stepped",
        "ljs_notation": f"{n}({terrace})x({step})",
        "ljs_candidates": candidates,
    })
    return selected


def generate_slab(
    metal,
    h,
    k,
    l,
    vacuum=15.0,
    width=10.0,
    fix=0.0,
    *,
    a=None,
    index=0,
    layer_tolerance=1e-3,
):
    """Build an orthogonal FCC metal(hkl) slab.

    `vacuum` is added on each side, `width` is the requested slab thickness,
    and `fix` is the constrained depth measured from the bottom-most atom.
    `a` optionally overrides the FCC lattice constant. Elemental FCC surfaces
    have one translationally unique cut, so `index` currently has to be zero.
    Terrace/step information is inferred from (h, k, l) and stored in
    `atoms.info["surface"]`. All distances are in angstrom.
    """
    metal = _normalize_metal(metal)
    miller = _normalize_miller(h, k, l)
    geometry = _surface_geometry_from_miller(miller)
    vacuum = _distance("vacuum", vacuum, positive=True)
    width = _distance("width", width, positive=True)
    fix = _distance("fix", fix)
    layer_tolerance = _distance("layer_tolerance", layer_tolerance, positive=True)
    if isinstance(index, bool) or not isinstance(index, Integral):
        raise ValueError("index must be an integer.")
    if index != 0:
        raise IndexError("index must be 0; this elemental FCC surface has one model.")

    fcc_primitive, a_used = _build_fcc_primitive(metal, a)
    primitive_miller = _fcc_primitive_miller(miller)
    two_layers = surface(
        fcc_primitive, primitive_miller, 2, vacuum=None, periodic=True
    )
    layer_spacing = float(np.ptp(two_layers.positions[:, 2]))
    generated_layers = ceil(width / layer_spacing) + 1
    atoms = surface(
        fcc_primitive,
        primitive_miller,
        generated_layers,
        vacuum=None,
        periodic=True,
    )
    atoms, orthogonal_transform = _make_orthogonal_surface(atoms)
    atoms.set_pbc((True, True, False))
    z_rotation_degrees = _rotate_ab_to_xy(atoms)
    atoms.wrap(pbc=(True, True, False))
    # ASE centering makes `vacuum` unambiguously the space on each side.
    atoms.center(vacuum=vacuum, axis=2)
    _check_orthogonal_axis_aligned_cell(atoms)

    layer_count = _count_atomic_layers(atoms, layer_tolerance)
    z = atoms.positions[:, 2]
    z_bottom = float(z.min())
    z_top = float(z.max())
    fixed_indices = []
    if fix > 0:
        fixed_indices = np.flatnonzero(z <= z_bottom + fix + 1e-8).tolist()
    if fixed_indices:
        atoms.set_constraint(FixAtoms(indices=fixed_indices))

    hkl = (
        "".join(str(value) for value in miller)
        if all(0 <= value <= 9 for value in miller)
        else ",".join(str(value) for value in miller)
    )
    atoms.info["surface"] = {
        "kind": geometry["kind"],
        "metal": metal,
        "crystal_structure": "fcc",
        "notation": f"{metal}({hkl})",
        "miller_index": miller,
        "primitive_miller_index": primitive_miller,
        "terrace_facet": geometry["terrace_facet"],
        "step_facet": geometry["step_facet"],
        "terrace_rows": geometry["terrace_rows"],
        "zone_axis": geometry["zone_axis"],
        "miller_formula": geometry["miller_formula"],
        "ljs_notation": geometry["ljs_notation"],
        "ljs_candidates": geometry["ljs_candidates"],
        "a": a_used,
        "a_source": "ASE default" if a is None else "user",
        "requested_width": width,
        "actual_atom_span": z_top - z_bottom,
        "vacuum_each_side": vacuum,
        "actual_bottom_vacuum": z_bottom,
        "actual_top_vacuum": float(atoms.cell.lengths()[2]) - z_top,
        "fixed_bottom_depth": fix,
        "index": 0,
        "model_count": 1,
        "layer_count": layer_count,
        "fixed_atom_count": len(fixed_indices),
        "cell_angles": tuple(float(angle) for angle in atoms.cell.angles()),
        "orthogonal_supercell_transform": orthogonal_transform.tolist(),
        "z_rotation_degrees": z_rotation_degrees,
        "cell_alignment": "a parallel +x; b parallel +y; c parallel +z; 90/90/90 degrees",
    }
    return atoms

---
## Examples and selected surface

The selected example is Pt(211). From `(2, 1, 1)`, the function identifies three-row (111) terraces separated by (100) steps. Change the element or Miller indices to generate another supported FCC surface.

In [ ]:
# Surface identity: Pt(211).
metal = "Pt"
h, k, l = 2, 1, 1

# All three geometry controls are distances in angstrom.
vacuum = 8.0  # Empty space both below and above the slab.
width = 6.0   # Requested slab thickness.
fix = 2.0     # Fix atoms within 2 A of the bottom-most atom.

# Optional controls.
a = None   # None -> use ASE's default FCC lattice constant.
index = 0  # Elemental FCC surfaces have one model; keep this at zero.

final_structure = generate_slab(
    metal, h, k, l, vacuum, width, fix,
    a=a,
    index=index,
)

# Further examples (uncomment as needed):
# pt111 = generate_slab("Pt", 1, 1, 1)
# cu100 = generate_slab("Cu", 1, 0, 0)
# pt331 = generate_slab("Pt", 3, 3, 1)
# pt310 = generate_slab("Pt", 3, 1, 0)
# pt551 = generate_slab("Pt", 5, 5, 1)
# pt511 = generate_slab("Pt", 5, 1, 1)
# pt320 = generate_slab("Pt", 3, 2, 0)

print(final_structure.info["surface"])
print(f"Atoms: {len(final_structure)}")
print("Cell (angstrom):")
print(final_structure.cell)

In [ ]:
print("Selected terrace/step:", final_structure.info["surface"]["ljs_notation"])
print("All matching descriptions:", final_structure.info["surface"]["ljs_candidates"])

### How to interpret and choose between surface models

`atoms.info["surface"]` is metadata attached to the returned ASE object. Most of its values describe how the slab was made; changing the dictionary afterward does **not** rebuild or modify the atoms. Keep the function inputs as the source of truth and regenerate the slab when you want a different physical model.

There are two different kinds of choice:

1. **Choose an LJS description for reporting.** `ljs_candidates` contains every supported terrace/step notation that describes the same macroscopic Miller plane. These are crystallographically equivalent interpretations, not different atomic slabs. `generate_slab` places the preferred interpretation first and copies it into `terrace_facet`, `step_facet`, `terrace_rows`, `zone_axis`, and `ljs_notation`. The preference follows the explicit order in `LJS_FAMILIES`. In particular:

(<i>n</i>, <i>n</i>, <i>n</i>&minus;2) &harr; M(S)[(<i>n</i>&minus;1)(111) &times; (110)].

Choose another candidate only when a paper, experiment, or naming convention uses that alternative notation; the atomic coordinates do not need to be regenerated.

```python
surface = final_structure.info["surface"]
for model_index, candidate in enumerate(surface["ljs_candidates"]):
    print(model_index, candidate)

# Select an alternative description for reporting or analysis only.
model_index = 0
chosen_ljs_model = surface["ljs_candidates"][model_index]
```

For example, Pt(221) can have more than one LJS description, but every entry in `ljs_candidates` still refers to the same Pt(221) orientation generated from `miller_index=(2, 2, 1)`. Pt(211) has one supported description: three-row (111) terraces with (100) steps.

2. **Choose a physically different slab through the inputs.** A clean elemental FCC crystal has one translationally unique termination for a given orientation, so this generator reports `model_count == 1` and requires `index=0`. The guarded `index` name is retained to keep this explicit; it is not a parameter that needs scanning. Different physical models come from changing `width`, `a`, or the in-plane size after generation. `vacuum` changes the simulation cell, while `fix` changes which atoms can move. For example:

```python
thin = generate_slab("Pt", 2, 1, 1, vacuum=8, width=6, fix=2)
thick = generate_slab("Pt", 2, 1, 1, vacuum=8, width=10, fix=2)
wide = thin.repeat((2, 2, 1))
```

When comparing energies, use the same calculator and convergence settings. Direct total-energy comparisons are meaningful only for models with compatible composition, atom count, and surface area; otherwise use an appropriately normalized surface energy.

| Setting or metadata | What it controls | New atomic model? |
| :--- | :--- | :---: |
| `ljs_candidates` / `ljs_notation` | Alternative names for the same Miller orientation | No |
| `index` / `model_count` | Explicitly confirms the single clean elemental-FCC model (`0` / `1`) | No |
| `width` | Slab thickness and number of atomic layers | Yes |
| `atoms.repeat((nx, ny, 1))` | Optional in-plane enlargement after generation | Yes |
| `a` | FCC lattice constant used to build the slab | Yes |
| `vacuum` | Empty space on each side | Changes the cell, not the surface structure |
| `fix` | Bottom region held fixed during relaxation | Changes the relaxation model |

In [ ]:
# Optional interactive view from another direction:
# view(final_structure, viewer="x3d")

---
## Isometric POV-Ray previews before optimization

The preview uses an **orthographic** camera so equal lengths retain equal scale and parallel cell edges remain parallel. Following the [isometric-projection guide](https://doublelayer.eu/vilab/2026/03/07/isometric-projection-for-scientific-graphics/), each rendering copy uses these rotations:

<i>&theta;</i><sub><i>x</i></sub> = 225&deg;, <i>&theta;</i><sub><i>y</i></sub> = 180&deg; + arctan(1/&radic;2) = 215.264&deg;, and <i>&theta;</i><sub><i>z</i></sub> = 30&deg;.

Rotating a copy prevents the rendering step from changing the structure that will later be optimized.

The camera, lighting, `intermediate` atom texture, antialiasing, and high-quality output follow the workflow in [Rendering Models with ASE and POV-Ray](https://github.com/doublelayer/ElectrocatalysisNotebooks/blob/main/1.3_Rendering_Models.ipynb). ASE creates the `.pov` and `.ini` files, while a checked subprocess runs POV-Ray quietly and reports useful diagnostics if rendering fails.

The loop below generates the requested vicinal-surface series from their exact Miller-index relations:

Pt(S)&minus;[(<i>n</i>&minus;1)(111) &times; (110)] &equiv; Pt(<i>n</i>, <i>n</i>, <i>n</i>&minus;2)

for <i>n</i> = 20, 14, 10, 7, 5, 4, 3, 2, and

Pt(S)&minus;[<i>n</i>(111) &times; (100)] &equiv; Pt(<i>n</i>+1, <i>n</i>&minus;1, <i>n</i>&minus;1)

for <i>n</i> = 21, 19, 15, 6, 4, 3, 2, 1. Each Miller triplet is reduced to the smallest integers before slab generation. For example, `(20, 20, 18)` is written as `(10, 10, 9)`. PNG filenames use those reduced indices, such as `Pt_10_10_9.png`. The structural models remain minimal cells; only temporary rendering copies are repeated until both in-plane dimensions are at least `render_minimum_inplane`, making narrow unit cells readable without putting `repeat` back into `generate_slab`.

POV-Ray is an external executable, not a Python package. Install it before running this section and ensure that `povray` is on `PATH` (for Debian/Ubuntu: `sudo apt install povray povray-includes`). The render test is deliberately placed before calculator setup and optimization: a missing renderer or failed image stops the notebook here. The `.pov`, `.ini`, and `.png` files are retained in `renders/` for inspection and reproducibility.

In [ ]:
ISOMETRIC_ROTATIONS = (
    (225.0, "x"),
    (180.0 + np.degrees(np.arctan(1.0 / np.sqrt(2.0))), "y"),
    (30.0, "z"),
)


def _isometric_copy(atoms):
    """Return a rotated copy; never change the model used for optimization."""
    rendered_atoms = atoms.copy()
    for angle, axis in ISOMETRIC_ROTATIONS:
        rendered_atoms.rotate(angle, axis, rotate_cell=True)
    return rendered_atoms


def render_isometric_povray(
    atoms,
    name,
    *,
    output_dir="renders",
    canvas_width=1600,
    radii=1.2,
    show_unit_cell=2,
    quality=11,
    povray_executable="povray",
):
    """Render one ASE model as an orthographic isometric PNG with POV-Ray."""
    executable = which(povray_executable)
    if executable is None:
        raise RuntimeError(
            f"POV-Ray executable {povray_executable!r} was not found on PATH. "
            "Install POV-Ray before starting the optimization."
        )
    if isinstance(canvas_width, bool) or not isinstance(canvas_width, Integral) or canvas_width < 1:
        raise ValueError("canvas_width must be a positive integer.")
    if isinstance(quality, bool) or not isinstance(quality, Integral) or not 0 <= quality <= 11:
        raise ValueError("quality must be an integer from 0 to 11.")

    safe_name = "".join(
        character if character.isalnum() or character in "-_" else "_"
        for character in str(name)
    )
    output_dir = Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    pov_path = output_dir / f"{safe_name}.pov"
    rendered_atoms = _isometric_copy(atoms)

    povray_settings = {
        "display": False,
        "pause": False,
        "transparent": False,
        "background": "White",
        "camera_type": "orthographic",
        "camera_dist": 1000.0,
        "canvas_width": int(canvas_width),
        "canvas_height": None,
        "image_plane": None,
        "depth_cueing": False,
        "point_lights": [],
        "area_light": [(0, 0, 100), "White", 1000, 1000, 1, 1],
        "celllinewidth": 0.03,
        "textures": ["intermediate"] * len(rendered_atoms),
    }
    renderer = write(
        pov_path,
        rendered_atoms,
        format="pov",
        radii=radii,
        colors=None,
        show_unit_cell=show_unit_cell,
        povray_settings=povray_settings,
    )

    # ASE writes only the POV basename into the INI file. Use absolute paths so an
    # output subdirectory works regardless of the notebook's current directory.
    ini_text = renderer.path.read_text(encoding="utf-8")
    ini_text = ini_text.replace(
        f"Input_File_Name={pov_path.name}",
        f'Input_File_Name="{pov_path}"',
    )
    png_path = pov_path.with_suffix(".png")
    ini_text += f'Output_File_Name="{png_path}"\nQuality={int(quality)}\n'
    renderer.path.write_text(ini_text, encoding="utf-8")

    completed = run(
        [executable, str(renderer.path)],
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        diagnostics = (completed.stdout + "\n" + completed.stderr).strip()
        raise RuntimeError(
            f"POV-Ray failed for {name!r} with exit code {completed.returncode}:\n"
            f"{diagnostics[-4000:]}"
        )
    if not png_path.is_file():
        raise RuntimeError(f"POV-Ray completed but did not create {png_path}.")
    return png_path

In [ ]:
FIRST_VICINAL_SERIES_N = (20, 14, 10, 7, 5, 4, 3, 2)
SECOND_VICINAL_SERIES_N = (21, 19, 15, 6, 4, 3, 2, 1)

render_before_optimization = True
render_minimum_inplane = 8.0  # Angstrom; applied only to preview copies.
render_output_dir = Path("renders")


def _miller_filename(metal, miller):
    """Return a readable filename stem from a reduced Miller triplet."""
    return f"{metal}_" + "_".join(str(value) for value in miller)


def _rendering_supercell(atoms, minimum_inplane):
    """Repeat a copy just enough to make both preview dimensions readable."""
    repeats = tuple(
        max(1, ceil(minimum_inplane / length))
        for length in atoms.cell.lengths()[:2]
    )
    return atoms.repeat((*repeats, 1)), repeats


render_specs = []
for n in FIRST_VICINAL_SERIES_N:
    render_specs.append(
        {
            "family": f"Pt(S)-[{n - 1}(111)x(110)]",
            "miller": _reduce_miller((n, n, n - 2)),
            "n": n,
        }
    )
for n in SECOND_VICINAL_SERIES_N:
    render_specs.append(
        {
            "family": f"Pt(S)-[{n}(111)x(100)]",
            "miller": _reduce_miller((n + 1, n - 1, n - 1)),
            "n": n,
        }
    )

models_to_render = {}
for spec in render_specs:
    miller = spec["miller"]
    model_name = _miller_filename("Pt", miller)
    if model_name in models_to_render:
        raise RuntimeError(f"Duplicate reduced Miller index: {miller}")
    model = generate_slab(
        "Pt", *miller, vacuum=vacuum, width=width, fix=fix, a=a, index=index
    )
    model.info["surface"]["series"] = spec["family"]
    model.info["surface"]["series_n"] = spec["n"]
    models_to_render[model_name] = model

rendered_model_paths = {}
if render_before_optimization:
    for spec, (model_name, model) in zip(render_specs, models_to_render.items()):
        hkl = " ".join(str(value) for value in spec["miller"])
        preview, repeats = _rendering_supercell(model, render_minimum_inplane)
        print(
            f"Rendering {spec['family']} as Pt({hkl}), preview repeat "
            f"{repeats} -> {model_name}.png"
        )
        png_path = render_isometric_povray(
            preview, model_name, output_dir=render_output_dir
        )
        rendered_model_paths[model_name] = png_path
        display(NotebookImage(filename=str(png_path), width=600))
else:
    raise RuntimeError("Render validation was skipped; enable it before optimization.")

print(f"POV-Ray validation passed for {len(rendered_model_paths)} model(s).")

---
## TBLite relaxation through ASE

This is a position-only BFGS relaxation. The cell and the bottom region selected by `fix` remain fixed. TBLite is a fast approximate model; convergence and scientific conclusions should be checked carefully for metallic surfaces.

In [ ]:
xtb_method = "GFN1-xTB"
charge = 0
multiplicity = None
electronic_temperature = 300.0
max_scf_iterations = 250
fmax = 0.05
max_geometry_steps = 200
run_relaxation = True
overwrite_outputs = False

surface_info = final_structure.info["surface"]
hkl = "".join(str(value) for value in surface_info["miller_index"])
if surface_info["kind"] == "LJS stepped":
    stem = (
        f"{surface_info['metal']}_{surface_info['terrace_rows']}_"
        f"{surface_info['terrace_facet']}x{surface_info['step_facet']}_{hkl}"
    )
else:
    stem = f"{surface_info['metal']}_{hkl}"
final_path = Path(f"{stem}.traj")
final_xyz_path = Path(f"{stem}.xyz")
optimization_path = Path(f"{stem}_optimization.traj")
optimization_log_path = Path(f"{stem}_optimization.log")
artifacts = (final_path, final_xyz_path, optimization_path, optimization_log_path)

existing = [path for path in artifacts if path.exists()]
if existing and not overwrite_outputs:
    names = ", ".join(str(path) for path in existing)
    raise FileExistsError(
        f"Refusing to overwrite existing calculation artifacts: {names}. "
        "Change the surface parameters or set overwrite_outputs=True."
    )

calculator_options = {
    "method": xtb_method,
    "charge": charge,
    "electronic_temperature": electronic_temperature,
    "max_iterations": max_scf_iterations,
    "verbosity": 1,
}
if multiplicity is not None:
    calculator_options["multiplicity"] = multiplicity

final_structure.calc = TBLite(**calculator_options)
initial_energy = final_structure.get_potential_energy()
initial_forces = final_structure.get_forces()
initial_fmax = np.linalg.norm(initial_forces, axis=1).max()
print(f"Initial TBLite energy: {initial_energy:.8f} eV")
print(f"Initial maximum movable-atom force: {initial_fmax:.6f} eV/angstrom")

In [ ]:
if run_relaxation:
    optimizer = BFGS(
        final_structure,
        trajectory=optimization_path,
        logfile=optimization_log_path,
    )
    converged = optimizer.run(fmax=fmax, steps=max_geometry_steps)
else:
    converged = initial_fmax <= fmax
    print("Relaxation skipped because run_relaxation=False.")

final_energy = final_structure.get_potential_energy()
final_forces = final_structure.get_forces()
final_fmax = np.linalg.norm(final_forces, axis=1).max()

print(f"Converged: {converged}")
print(f"Final TBLite energy: {final_energy:.8f} eV")
print(f"Energy change: {final_energy - initial_energy:+.8f} eV")
print(f"Final maximum movable-atom force: {final_fmax:.6f} eV/angstrom")

---
## Save and inspect

The ASE trajectory is the primary reusable structure because it preserves the cell, constraints, metadata, and calculator results. An extended XYZ copy is also written for convenient exchange and visualization.

In [ ]:
write(final_path, final_structure)
write(final_xyz_path, final_structure, format="extxyz")

saved_structure = read(final_path)
if len(saved_structure) != len(final_structure):
    raise RuntimeError("Saved structure failed the atom-count integrity check.")
if not np.allclose(saved_structure.cell.array, final_structure.cell.array):
    raise RuntimeError("Saved structure failed the cell integrity check.")

print(f"Saved ASE trajectory: {final_path.resolve()}")
print(f"Saved extended XYZ:  {final_xyz_path.resolve()}")
if run_relaxation:
    print(f"Optimization history: {optimization_path.resolve()}")
    print(f"Optimization log:     {optimization_log_path.resolve()}")

print(f"Viewing final {final_structure.info['surface']['notation']}")
view(final_structure)